In [ ]:
!git clone https://github.com/Morteza-24/llm-uncertainty-head.git
!mv llm-uncertainty-head/* llm-uncertainty-head/.[!.]* ./
%pip install git+https://github.com/IINemo/lm-polygraph.git
%pip install -e .

Cloning into 'llm-uncertainty-head'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 260 (delta 58), reused 67 (delta 46), pack-reused 155 (from 1)
Receiving objects: 100% (260/260), 666.35 KiB | 13.60 MiB/s, done.
Resolving deltas: 100% (137/137), done.
  Cloning https://github.com/IINemo/lm-polygraph.git to /tmp/pip-req-build-nohdw25q
  Running command git clone --filter=blob:none --quiet https://github.com/IINemo/lm-polygraph.git /tmp/pip-req-build-nohdw25q
  Resolved https://github.com/IINemo/lm-polygraph.git to commit efea882d810d07770e71d3a80e02416d09751435
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 15.7 MB/

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from luh import AutoUncertaintyHead
from lm_polygraph import CausalLMWithUncertainty
from luh.calculator_infer_luh import CalculatorInferLuh
from luh.luh_estimator_dummy import LuhEstimatorDummy

In [2]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
uhead_name = "llm-uncertainty-head/uhead6_Mistral-7B-Instruct-v0.2"

llm = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(
    model_name)
tokenizer.pad_token = tokenizer.eos_token
uhead = AutoUncertaintyHead.from_pretrained(
    uhead_name, base_model=llm)

generation_config = GenerationConfig.from_pretrained(model_name)
args_generate = {"generation_config": generation_config,
                 "max_new_tokens": 50}
calc_infer_llm = CalculatorInferLuh(uhead,
                                    tokenize=True,
                                    args_generate=args_generate,
                                    device="cuda",
                                    generations_cache_dir="",
                                    predict_token_uncertainties=True)

estimator = LuhEstimatorDummy()
llm_adapter = CausalLMWithUncertainty(llm, tokenizer=tokenizer, stat_calculators=[calc_infer_llm], estimator=estimator)

# prepare text ...
messages = [
    [
        {
            "role": "user",
            "content": "In which year did the programming language Mercury first appear? Answer with a year only."
        }
    ]
]
# The correct answer is 1995
chat_messages = [tokenizer.apply_chat_template(m, tokenize=False, add_bos_token=False) for m in messages]
inputs = tokenizer(chat_messages, return_tensors="pt", padding=True, truncation=True, add_special_tokens=False).to("cuda")

output = llm_adapter.generate(inputs["input_ids"])
output["uncertainty_score"]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

weights.pth: reconstructing file:   0%|          |  0.00B / 19.7MB            

weights.pth: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=46) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[[0.5005925893783569,
  0.11431393772363663,
  0.19080078601837158,
  0.6520728468894958,
  0.430184930562973,
  0.12376189231872559,
  0.3136047422885895,
  0.5520869493484497,
  0.4270686209201813,
  0.36228707432746887,
  0.5659871101379395,
  0.814650297164917,
  0.6513360738754272,
  0.7492914199829102,
  0.7665389776229858,
  0.8988367319107056,
  0.7635713219642639,
  0.7144246697425842,
  0.5759887099266052,
  0.7082247734069824]]